In [ ]:
import os
import numpy as np
import nibabel as nib
import pandas as pd
from tqdm import tqdm

SALIENCY_DIR = "/media/ravi-bullock/Elements/VoxelWiseSaliency/AdultSynthetic/Individual_Saliency_Maps_Age65"
SEG_DIR = "AdultSynthetic/Segmentations"
MASK_DIR = "AdultSynthetic/Masks"
OUT_CSV = "Adult_Age12_Final_Metrics.csv"

target_rois = {
    "Left_Thalamus": [91],
    "Right_Thalamus": [40],
    "Left_Lateral_Ventricle": [92],
    "Right_Lateral_Ventricle": [41],
}


def calculate_occupancy(grad_map, roi_mask, brain_mask):
    """Ratio of TOTAL saliency mass: ROI vs Whole Brain."""
    total_brain_sal = np.sum(grad_map[brain_mask > 0])
    if total_brain_sal <= 0: return 0
    sal_in_roi = np.sum(grad_map[roi_mask > 0])
    return sal_in_roi / total_brain_sal

def calculate_rsm(grad_map, roi_mask, brain_mask):
    """Ratio of MEAN saliency intensity: ROI vs Non-ROI Brain Tissue."""
    inside_pts = grad_map[roi_mask > 0]
    outside_pts = grad_map[(brain_mask > 0) & (roi_mask == 0)]
    
    inside_mean = np.mean(inside_pts) if inside_pts.size > 0 else 0
    outside_mean = np.mean(outside_pts) if outside_pts.size > 0 else 1e-8
    
    return inside_mean / (outside_mean + 1e-12)

results = []
sal_files = [f for f in os.listdir(SALIENCY_DIR) if f.endswith("_saliency.nii.gz")]

if not sal_files:
    print(f"❌ Error: No saliency files found in {SALIENCY_DIR}")
else:
    for sal_fname in tqdm(sal_files, desc="Processing Metrics"):
        matched_roi = None
        for r_name in target_rois.keys():
            if r_name in sal_fname:
                matched_roi = r_name
                break
        
        if not matched_roi:
            continue

        sub_id = sal_fname.split(f"_{matched_roi}")[0]
        
        seg_path = os.path.join(SEG_DIR, f"{sub_id}.nii.gz") 
        mask_path = os.path.join(MASK_DIR, f"{sub_id}.nii.gz")
        
        if not (os.path.exists(seg_path) and os.path.exists(mask_path)):
            seg_path = os.path.join(SEG_DIR, f"{sub_id}_T1w.nii.gz")
            mask_path = os.path.join(MASK_DIR, f"{sub_id}_T1w.nii.gz")
            if not os.path.exists(seg_path):
                continue

        grad_map = nib.load(os.path.join(SALIENCY_DIR, sal_fname)).get_fdata()
        seg_data = nib.load(seg_path).get_fdata()
        brain_mask = nib.load(mask_path).get_fdata()

        roi_mask = np.isin(seg_data, target_rois[matched_roi])

        occ = calculate_occupancy(grad_map, roi_mask, brain_mask)
        rsm = calculate_rsm(grad_map, roi_mask, brain_mask)

        results.append({
            "Subject": sub_id,
            "ROI": matched_roi,
            "Fractional_Occupancy": occ,
            "RSM": rsm
        })

if not results:
    print("❌ No results collected.")
else:
    df = pd.DataFrame(results)
    
    summary = df.groupby("ROI")[["Fractional_Occupancy", "RSM"]].agg(['mean', 'std'])
    print("\n--- Final Adult Saliency Metrics ---")
    print(summary)
    
    df.to_csv(OUT_CSV, index=False)
    print(f"\n✅ Results saved to {OUT_CSV}")